In [2]:
!pip install gdown

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [4]:
!gdown --folder -O data "1igPWZSzmpZE5EROIQkgirhmow2UZ44IX"


Retrieving folder contents
Retrieving folder 1fcGozwyV4Y9Peh9mhWpa7dQ50FnA3qDc old_v1
Processing file 1RdRUtpy1rSoBn8_iFtwYb69tvW2uzl4A val.jsonl
Processing file 1tckBTOCH0Zc28M5Y6ywpp3UvAbHbLyPU train.jsonl
Processing file 1a-h08V6NZR8tFF2rNWsib_ec8-I9TgIo train_pairs.jsonl
Processing file 1gj4J-04V0tzrUOnTpaHG5hehkG-H1IVF train_triplets.jsonl
Processing file 1zEhdtaNEJsujMBwiXJAs98N2WMvPqU13 val.jsonl
Processing file 17Bcjp9WvKHHTWfUl6oc29m_XQZokgjI5 val_pairs.jsonl
Processing file 1NjEc9TNa6540sNi0bvvpHcSqBydP5vXJ val_triplets.jsonl
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1RdRUtpy1rSoBn8_iFtwYb69tvW2uzl4A
To: /home/jovyan/long_context_ropg/data/old_v1/val.jsonl
100%|██████████████████████████████████████| 2.71M/2.71M [00:02<00:00, 1.32MB/s]
Downloading...
From: https://drive.google.com/uc?id=1tckBTOCH0Zc28M5Y6ywpp3UvAbHbLyPU
To: /home/jovyan/long_context_ropg/data/tra

In [5]:
!pip install torch sentence-transformers peft pyyaml tqdm

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [6]:
!pip install transformers datasets accelerate peft

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 1.8 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 1.9 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1
  Attempting uninstall: dill
    Found existing installation: dill 0.3.9
    Uninstalling dill-0.3.9:
      Successfully uninstalled dill-0.3.9


In [10]:
# !pip install torchao

In [7]:
!pip install --no-deps --upgrade torchao>=0.16.0

In [6]:
# pip install torch==2.11.0  --index-url https://download.pytorch.org/whl/cu128

In [8]:
!mv ./data/train.jsonl ./data/train_scored.jsonl
!mv ./data/val.jsonl ./data/val_scored.jsonl

In [8]:
# !python /kaggle/input/datasets/fake1colabgpu/train-embed-ddp/train_embedding_ddp.py     --model_name Qwen/Qwen3-Embedding-0.6B     --train_data /kaggle/working/ropg_kd     --output_dir ./qwen3-embed-ropg-kd     --mode reader_kd     --num_epochs 3     --batch_size 2     --learning_rate 2e-5     --temperature 1.0     --max_documents 20     --use_amp  --eval_top_k 5 --eval_batch_size 1  --best_metric mrr

In [12]:
!torchrun --nproc_per_node=1 ./train_embedding_ddp_chunk_final.py \
    --model_name Qwen/Qwen3-Embedding-0.6B \
    --train_data ./data \
    --output_dir ./qwen3-embed-ropg \
    --mode reader_kd \
    --num_epochs 3 \
    --batch_size 1 \
    --max_length 4000 \
    --use_lora \
    --use_amp \
    --chunk_size 2048 \
    --chunk_overlap 300 \
    --max_documents 26   # optional: allow more chunks per query

Failed to load /opt/conda/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /opt/conda/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /opt/conda/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /opt/conda/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Device : cuda:0
Model  : Qwen/Qwen3-Embedding-0.6B
Mode   : reader_kd
Format : triplets
Max length: 4000
LoRA    : True
World size: 1
Chunking enabled: size=2048, overlap=300
Train samples: 1293
Val samples  : 276
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 310/310 [00:00<00:00, 3203.07it/s]
trainable params: 2,293,760 || all params: 598,070,272 || trainable%: 0.3835
Total parameters: 598,070,272
Steps/epoch  : 1293
Total steps  : 3879
Warmup steps : 387

Best model selection based on: val_loss (goal: min)

[transform

In [ ]:
!torchrun --nproc_per_node=1 ./train_embedding_ddp_wo_chunk.py \
    --model_name Qwen/Qwen3-Embedding-0.6B \
    --train_data ./data \
    --output_dir ./qwen3-embed-ropg \
    --mode reader_kd \
    --num_epochs 3 \
    --batch_size 1 \
    --max_length 2048 \
    --learning_rate 2e-5 \
    --eval_top_k 5 \
    --best_metric recall@1 \
    --use_amp \
    --use_lora \
    # --lora_r 4

    # --mode hard_neg \
    # --format triplets \

Failed to load /opt/conda/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /opt/conda/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /opt/conda/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /opt/conda/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Device : cuda:0
Model  : Qwen/Qwen3-Embedding-0.6B
Mode   : reader_kd
Format : triplets
Max length: 2048
LoRA    : True
World size: 1
Train samples: 1293
Val samples  : 276
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 310/310 [00:00<00:00, 2635.78it/s]
trainable params: 2,293,760 || all params: 598,070,272 || trainable%: 0.3835
Total parameters: 598,070,272
Steps/epoch  : 1293
Total steps  : 3879
Warmup steps : 387

Best model selection based on: recall@1 (goal: max)

[transformers] `use_cache=True` is incompatible wit

In [ ]:
# --lora_r

In [ ]:
# !torchrun --nproc_per_node=2 /kaggle/input/datasets/fake1colabgpu/ddp-chunk-correct-2/train_embedding_ddp_chunk.py \
#     --model_name Qwen/Qwen3-Embedding-0.6B \
#     --train_data /kaggle/working/ropg_kd \
#     --output_dir ./qwen3-embed-ropg \
#     --mode reader_kd \
#     --num_epochs 3 \
#     --batch_size 1 \
#     --gradient_accumulation_steps 4 \
#     --max_length 4096 \
#     --learning_rate 2e-5 \
#     --eval_top_k 5 --best_metric recall@1 \
#     --use_lora --use_amp \
#     --train_chunk_size 512 \
#     --eval_chunk_size 512 \
#     --max_chunks_per_sample 7 \
#     --micro_batch_size 8 \
#     --num_workers 1

#     # --mode hard_neg \
#     # --format triplets \

Here are separate run commands for each data format, using the fixed DDP script with `torchrun` and 2× T4 GPUs.

---

### 1. **Hard Negative Training – Triplets**
```bash
torchrun --nproc_per_node=2 train_embedding_ddp.py \
    --model_name Qwen/Qwen3-Embedding-0.6B \
    --train_data ./ropg_dataset \
    --output_dir ./qwen3-embed-ropg \
    --mode hard_neg \
    --format triplets \
    --num_epochs 3 \
    --batch_size 4 \
    --max_length 4096 \
    --learning_rate 2e-5 \
    --eval_top_k 5 \
    --best_metric recall@1 \
    --use_lora \
    --use_amp
```

**Notes:**
- The script expects `train_triplets.jsonl` and `val_triplets.jsonl` inside `./ropg_dataset`.
- `--format triplets` tells the script to use those files.

---

### 2. **Hard Negative Training – Pairs (expanded negative)**
```bash
torchrun --nproc_per_node=2 train_embedding_ddp.py \
    --model_name Qwen/Qwen3-Embedding-0.6B \
    --train_data ./ropg_dataset \
    --output_dir ./qwen3-embed-ropg-pairs \
    --mode hard_neg \
    --format pairs \
    --num_epochs 3 \
    --batch_size 4 \
    --max_length 4096 \
    --learning_rate 2e-5 \
    --eval_top_k 5 \
    --best_metric recall@1 \
    --use_lora \
    --use_amp
```

**Notes:**
- Expects `train.jsonl` and `val.jsonl` (each line has `query`, `positive`, `negative`).
- The script internally treats each pair as a triplet with one negative.

---

### 3. **Knowledge Distillation – Scored Data**
```bash
torchrun --nproc_per_node=2 train_embedding_ddp.py \
    --model_name Qwen/Qwen3-Embedding-0.6B \
    --train_data ./ropg_dataset \
    --output_dir ./qwen3-embed-ropg-kd \
    --mode reader_kd \
    --num_epochs 3 \
    --batch_size 4 \
    --max_length 4096 \
    --learning_rate 2e-5 \
    --temperature 1.0 \
    --max_documents 20 \
    --eval_top_k 10 \
    --best_metric ndcg@10 \
    --use_lora \
    --use_amp
```

**Notes:**
- Uses `train_scored.jsonl` and `val_scored.jsonl` – the script expects the scored format (with `docs` array and `teacher_score`).
- `--format` is ignored in `reader_kd` mode.
- `--temperature` controls the softmax temperature for KL divergence.
- `--max_documents` limits the number of documents per query in the dataset.

---

### Common parameters explained
| Argument | Description |
|----------|-------------|
| `--batch_size 4` | Safe for 4096‑token chunks on 2× T4 (16 GB each) – increase to 8 if VRAM allows. |
| `--max_length 4096` | Matches your chunk length (you mentioned ~4000). |
| `--use_lora` | Reduces VRAM usage drastically (~4‑6 GB per GPU) – recommended. |
| `--use_amp` | Enables bfloat16 mixed precision – faster and lower memory. |
| `--eval_top_k` | K for Recall@K and MRR/NDCG. |
| `--best_metric` | Metric used to select the best checkpoint (e.g., `recall@1`, `mrr`, `ndcg@10`). |
| `--num_epochs 3` | Adjust as needed. |

---

### Before running (on Kaggle)
Make sure you've installed the required packages and fixed the `torchao` version:

```python
!pip install -q transformers datasets accelerate peft
!pip install --no-deps --upgrade torchao>=0.16.0   # Fix for peft/torchao conflict
```

Then run any of the above commands in a notebook cell (prefixed with `!`).